# AI-Powered Internship Cold Email Outreach System

This notebook demonstrates how to build a multi-agent workflow for generating, formatting, and sending personalized cold outreach emails for software engineering internships.

### Key Features:
- **Multi-Agent Collaboration**: Diverse agents collaborate to draft covers, pick the best ones, format them to HTML, and send them.
- **Multi-Provider Support**: Seamlessly run different agents using models from **OpenAI**, **Gemini**, or **Groq** via OpenAI-compatible SDK clients.
- **Gradio Interface**: A user-friendly web interface to generate, review, and send outreach campaigns.

## 1. Imports and Environmental Setup

First, we import the required dependencies (including our custom `agents` library, `sendgrid` for mailing, and `gradio` for the web UI) and load the API keys from the `.env` file.

In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
import asyncio
import gradio as gr
import json
from openai import OpenAI



In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')


if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")

## 2. Multi-Provider Client Initialization

To support diverse model endpoints, we leverage OpenAI-compatible APIs for Gemini and Groq. We instantiate standard OpenAI clients pointing to the respective provider base URLs and wrap them in `OpenAIChatCompletionsModel` for agent compatibility.

In [ ]:
openai = OpenAI()

groq_url = "https://api.groq.com/openai/v1"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

gemini_model = OpenAIChatCompletionsModel(
    model="gemini-2.5-flash-lite", 
    openai_client=gemini
)

groq_model = OpenAIChatCompletionsModel(
    model="llama-3.2-3b-preview", 
    openai_client=groq
)


## 3. Email Integration (SendGrid Verification)

We verify that our SendGrid API key and verified sender domain are working correctly by sending a simple test email.

In [ ]:

def send_test_email():
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("example@gmail.com")  # Change to your verified sender
    to_email = To("example@gmail.com")  # Change to your recipient
    content = Content("text/plain", "This is an important test email")
    mail = Mail(from_email, to_email, "Test email", content).get()
    response = sg.client.mail.send.post(request_body=mail)
    print(response.status_code)

send_test_email()

## 4. Multi-Agent Setup & Personas

In this step, we configure three different copywriter personas for generating diverse cover letters, along with a picker agent that evaluates and selects the best candidate letter.

In [ ]:
instructions1 = "You are a professional Engineering intern candidate applying to early-stage startups. \
You write strong, professional cover letters that demonstrate technical skills, problem-solving ability, initiative, and how you can help the startup build AI-powered products quickly. \
Your tone is confident, serious, and startup-focused."

instructions2 = "You are an ambitious and engaging Engineering intern candidate applying to early-stage startups. \
You write witty, energetic, and memorable cover letters that showcase passion for AI, building products, experimenting with new technologies, and helping startups move fast. \
Your writing feels human, enthusiastic, and founder-friendly while still remaining professional."

instructions3 = "You are a busy Engineering intern candidate applying to early-stage startups. \
You write concise, direct, and impactful cover letters that quickly explain technical skills, relevant AI projects, and how you can immediately contribute to the startup. \
Your writing is short, clear, and highly focused on value."

In [ ]:
cover_letter_picker = Agent(
    name="cover_letter_picker",
    instructions="You pick the best software development internship cover letter from the given options. \
Imagine you are the founder of as tartup hiring an intern and select the cover letter most likely to get an interview. \
Prioritize clarity, technical relevance, enthusiasm, startup mindset, and demonstrated value. \
Do not give an explanation; reply with the selected cover letter only.",
    model="gpt-5-mini"
)

### Candidate & Recipient Details

Fill in your own applicant details below before executing the cells.

In [ ]:
company_name = "OpenAI Labs"

recipient_email = "founder@openailabs.ai"

company_website = "https://openailabs.ai"

linkedin_url = "https://linkedin.com/in/sam-altman"

candidate_name = "your-name"
candidate_email = " sender email"  # sender email (must be verified in SendGrid)

candidate_linkedin = "https://www.linkedin.com/in/username/"#add your username here or replace the link.

candidate_github = "https://github.com/username"#add your username here or replace the link.


### Instantiate the Writing Agents

We instantiate each writer agent using our configured models from different providers (OpenAI, Groq, and Gemini).

In [ ]:
cover_letter_agent1 = Agent(
        name="Professional Intern Applicant",
        instructions=instructions1,
        model="gpt-5-nano" #you can also use groq_model or gemini_model here
)




cover_letter_agent2 = Agent(
        name="Engaging Intern Applicant",
        instructions=instructions2,
        model=groq_model
)

cover_letter_agent3 = Agent(
        name="Concise Intern Applicant",
        instructions=instructions3,
        model=gemini_model
)

## 5. Tools & Agent-to-Tool Conversions

### What are Tools?
**Tools** represent actions or routines that an agent can invoke during execution. By providing tools to an agent, you extend its capabilities beyond pure text generation. 

There are two primary ways we create tools here:
1. **Function Tools (`@function_tool`)**: Turns a Python function (like `send_email`) into a tool by parsing its docstring and type hints to generate a JSON schema that the LLM understands.
2. **Agent Tools (`agent.as_tool(...)`)**: Converts an existing agent into a tool. This allows a master orchestrator agent to "call" other agents to delegate specific drafting sub-tasks.

In [ ]:
@function_tool
def send_email(body: str, recipient_email: str):
    """ Send out an email with the given body to all cover letter agent prospects """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email(candidate_email)  # Change to your verified sender
    to_email = To(recipient_email)  # Change to your recipient
    content = Content("text/plain", body)
    mail = Mail(from_email, to_email, "Application", content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [ ]:
tool1 = cover_letter_agent1.as_tool(tool_name="sales_agent1", tool_description="Write a cold sales email")
tool1

In [ ]:
description = "Write a cold personalised email"

tool1 = cover_letter_agent1.as_tool(tool_name="cover_letter_agent1", tool_description=description)
tool2 = cover_letter_agent2.as_tool(tool_name="cover_letter_agent2", tool_description=description)
tool3 = cover_letter_agent3.as_tool(tool_name="cover_letter_agent3", tool_description=description)

tools = [tool1, tool2, tool3, send_email]

tools

## 6. Formatting & Delivery Sub-workflow

Before sending out any emails, we construct professional email templates. Here, we establish specialized agents to write outreach subject lines and compile plain text into beautifully formatted HTML.

In [ ]:
subject_instructions = """
You write compelling subject lines for cold outreach emails sent to early-stage startups for Engineering Internship opportunities.
Your subject lines should feel personalized, concise, professional, and likely to get a response from a founder or hiring manager.
"""

html_instructions = """
You convert plain text cold outreach emails into clean HTML email templates.
The emails are for Engineering Internship applications to early-stage startups.
Create a minimal, modern, and professional HTML layout that is easy to read and founder-friendly.
"""

subject_writer = Agent(
    name="Internship Email Subject Writer",
    instructions=subject_instructions,
    model="gpt-5-mini" #you can also use groq_model or gemini_model here
)

subject_tool = subject_writer.as_tool(
    tool_name="subject_writer",
    tool_description="Write a subject line for an internship cold outreach email"
)

html_converter = Agent(
    name="HTML Email Converter",
    instructions=html_instructions,
    model="gpt-5-mini" #you can also use groq_model or gemini_model here
) 

html_tool = html_converter.as_tool(
    tool_name="html_converter",
    tool_description="Convert a cold outreach email into a clean HTML email template"
)

In [ ]:
@function_tool
def send_html_email(subject: str, html_body: str, recipient_email: str, sender_email: str = None) -> Dict[str, str]:
    """Send out an email with the given subject and HTML body."""
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email(candidate_email)  # must be verified in SendGrid
    to_email = To(recipient_email)
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    response = sg.client.mail.send.post(request_body=mail)
    return {"status": "success", "status_code": str(response.status_code)}

In [ ]:
tools = [subject_tool, html_tool, send_html_email]

In [ ]:
tools

### Define the Internship Email Manager Agent

This agent acts as a sub-workflow controller. It takes the selected email body, formats it to HTML via the `html_converter` tool, writes a subject line via the `subject_writer` tool, and invokes `send_html_email` to complete the transaction.

In [ ]:
instructions = """
You are an email formatter and sender for an engineering internship outreach emails.

Input format (you will receive this text):
- A JSON object with keys: recipient_email, body

Your workflow:
1. Use the subject_writer tool to generate a compelling subject line for `body`.
2. Use the html_converter tool to convert `body` into a clean HTML email.
3. Use the send_html_email tool with recipient_email + subject + html_body.

Rules:
- Always generate the subject first.
- Always convert the email to HTML before sending.
- Send only one final email.
- Do not ask questions; just do the workflow.
"""

emailer_agent = Agent(
    name="AI Internship Email Manager",
    instructions=instructions,
    tools=tools,
    model="gpt-5-nano",
    handoff_description="Generate subject lines, format internship outreach emails into HTML, and send them"
)

## 7. Master Orchestrator & Handoffs (The Application Manager)

### What are Handoffs?
**Handoffs** enable multi-agent systems to transfer control dynamically from one agent to another. 

- While **Tools** allow agents to execute structured function calls or query sub-agents synchronously, **Handoffs** completely transfer conversational context to another agent.
- We register handoffs by adding the target agent to the `handoffs=[target_agent]` list. The calling agent uses the target's `handoff_description` to decide when and why to route context to that agent.

### How Handoffs work here:
- The `application_manager` receives the initial request.
- It runs `cover_letter_agent1`, `cover_letter_agent2`, and `cover_letter_agent3` to generate variations.
- If `MODE` is set to `GENERATE_AND_SEND`, the orchestrator invokes a **handoff** to transfer control to `emailer_agent` along with a payload containing the recipient email and selected body.

In [ ]:
tools = [tool1, tool2, tool3]
handoffs = [emailer_agent]
print(tools)
print(handoffs)

In [ ]:
application_manager_instructions = """
You are an Internship Application Manager.

Your goal is to find the single best cold outreach email for an Engineering Internship using the cover_letter_agent tools.

You will receive a message containing:
- Job description
- Recipient email
- MODE (either GENERATE_ONLY or GENERATE_AND_SEND)

Steps:
1. Generate 3 different outreach emails using all three cover_letter_agent tools.
2. Review and select the strongest email based on personalization, technical relevance, startup fit, and response potential.
3. If needed, regenerate drafts until satisfied.

Mode rules:
- If MODE=GENERATE_ONLY: do NOT hand off. Reply with the selected email body only.
- If MODE=GENERATE_AND_SEND: hand off ONLY the final selected email to the 'AI Internship Email Manager' for formatting and sending.
  When handing off, pass a JSON payload with exactly:
  {"recipient_email": string, "body": string}
  Where `body` is the selected best email text.
  After the handoff, also reply with the selected email body only.

Global rules:
- Always use the cover_letter_agent tools.
- Never write the emails yourself.
"""

application_manager = Agent(
    name="AI Internship Application Manager",
    instructions=application_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model="gpt-5-nano"
)

# (Optional) example run:
#with trace("Automated AI Internship Outreach"):
     #result = await Runner.run(application_manager, "MODE=GENERATE_ONLY\n...")

## 8. Web Frontend (Gradio UI)

Finally, we launch an interactive interface using Gradio. This enables you to enter job details, preview outputs, choose generation or sending mode, and track agent traces in real time.

In [ ]:
def _build_manager_message(job_description: str, recipient_email: str, mode: str) -> str:
    company_hint = recipient_email.split("@")[-1] if "@" in recipient_email else recipient_email
    return f"""MODE={mode}

Recipient email: {recipient_email}
Company hint (from recipient email domain): {company_hint}

Job description:
{job_description}

Candidate:
Name: {candidate_name}
Email (sender): {candidate_email}
LinkedIn: {candidate_linkedin}
GitHub: {candidate_github}
"""


async def ui_run_manager(job_description: str, recipient_email: str, mode: str) -> str:
    msg = _build_manager_message(job_description, recipient_email, mode)
    with trace(f"Gradio: {mode}"):
        result = await Runner.run(application_manager, msg)
    return getattr(result, "final_output", "") or ""


with gr.Blocks(title="Internship Outreach") as demo:
    gr.Markdown("## Internship cold email\nGenerate the best email, or send the last generated email (uses subject + HTML + SendGrid tools).")

    recipient = gr.Textbox(label="Recipient email", placeholder="founder@company.com")
    jd = gr.Textbox(label="Job description", lines=10, placeholder="Paste job description...")

    with gr.Row():
        btn_generate = gr.Button("Generate email")
        btn_send = gr.Button("Send")

    last_email_state = gr.State("")

    out = gr.Textbox(label="Output", lines=18)

    async def ui_generate_only(job_description: str, recipient_email: str):
        email_body = await ui_run_manager(job_description, recipient_email, "GENERATE_ONLY")
        return email_body, email_body

    async def ui_send_last_or_generate(job_description: str, recipient_email: str, last_email: str):
        email_body = last_email
        if not (email_body or "").strip():
            email_body = await ui_run_manager(job_description, recipient_email, "GENERATE_ONLY")

        payload = json.dumps({"recipient_email": recipient_email, "body": email_body}, ensure_ascii=False)
        with trace("Gradio: SEND_ONLY"):
            await Runner.run(emailer_agent, payload)

        return email_body, email_body

    btn_generate.click(fn=ui_generate_only, inputs=[jd, recipient], outputs=[out, last_email_state])
    btn_send.click(fn=ui_send_last_or_generate, inputs=[jd, recipient, last_email_state], outputs=[out, last_email_state])


demo.queue()
demo.launch()